# 안녕하세요^^ 
# AIVLE 미니 프로젝트에 오신 여러분을 환영합니다.
* 본 과정에서는 실제 사례와 데이터를 기반으로 문제를 해결하는 전체 과정을 자기 주도형 실습으로 진행해볼 예정입니다.
* 앞선 교육과정을 정리하는 마음과 지금까지 배운 내용을 바탕으로 문제 해결을 해볼게요!
* 미니 프로젝트를 통한 문제 해결 과정 'A에서 Z까지', 지금부터 시작합니다!

---

## 0. 환경 설정하기

### 1) 구글 드라이브 연결하기

In [27]:
# 코랩 사용 시 구글 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 2) 경로 확인하기
- "WORK_SPACE" 에 본인 작업 경로 작성 후 실행(구글 드라이브 최상위에 압축해제 시 그대로 실행. 수정 X).<br>

<font color="red">※ 주의. 나머지 경로는 절대 변경하지 마세요.</font>

In [28]:
# ROOT_PATH 확인 
import os

# 구글 드라이브 내 프로젝트 압축해제된 영역 (구글 드라이브 최상위에 압축해제 시 그대로 실행 수정 X)
WORK_SPACE = "Colab Notebooks/aivle/미니프로젝트3"

if os.getcwd() == '/content' :
  # 구글 드라이브 사용 시 
  ROOT_PATH = "/content/drive/MyDrive/"+WORK_SPACE+"/AIVLE3rd_individual"
else :
  ROOT_PATH = os.path.abspath('..')
# Train 데이터 셋 경로
TRAIN_PATH = ROOT_PATH + "/train"
# MODEL 저장 경로
MODEL_PATH = ROOT_PATH + "/model"

### 3) 라이브러리 불러오기
필요시 추가 라이브러리는 설치해서 사용하세요.

In [3]:
# 필요 라이브러리 불러오기.
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Embedding
from keras.layers import Conv2D, MaxPooling2D, Flatten
from keras.callbacks import ModelCheckpoint, EarlyStopping

### 4) GPU 환경 확인하기
tensorflow가 GPU를 활용하고 있는지 확인하려면, tensorflow에서 제공하는 device_lib 라이브러리를 활용하면 됩니다.

In [4]:
# GPU 환경 확인하기
from tensorflow.python.client import device_lib
device_lib.list_local_devices()

[name: "/device:CPU:0"
 device_type: "CPU"
 memory_limit: 268435456
 locality {
 }
 incarnation: 6701299758208618892
 xla_global_id: -1, name: "/device:GPU:0"
 device_type: "GPU"
 memory_limit: 14444920832
 locality {
   bus_id: 1
   links {
   }
 }
 incarnation: 9074187553066469287
 physical_device_desc: "device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5"
 xla_global_id: 416903419]

In [5]:
import os
import shutil

---

# 1. ImageDataGenerator 생성하기
앞의 <font color="red">__'[study] 2.데이터전처리'__ </font>과정에서 사용하였던 ImageDataGenerator를 그대로 가져오시면 됩니다.

<font color="green">[실습문제]</font> 1. ImageDataGenerator 생성하기
+ 모델 검증을 위해 데이터를 train:validation(8:2)로 분할합니다.
+ 모델 성능 개선을 위해 데이터 증식(Data augmentation)이 필요 시 자유롭게 설정

In [55]:
# 실습해보세요.
train_datagen = ImageDataGenerator(
    rescale=1./255, # 0에서 1 로 만들기
    validation_split=0.2, 
)
batch_size=16
img_height=200
img_width=400
# train_genrator 생성
train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,# train 경로
    subset='training', #8:2로 나누기
    batch_size=8,
    color_mode = "grayscale", #<=======color_mode 설정
    target_size=(img_height,img_width),
    class_mode='categorical' #클래스 총 4개
)


# validation_generator 생성
validation_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    subset='validation',
    batch_size=8,color_mode = "grayscale",
    target_size=(img_height,img_width), # train과 동일하게 해야함
    class_mode='categorical'
)

Found 695 images belonging to 4 classes.
Found 172 images belonging to 4 classes.


---

# 2. 모델 구성하기
+ KeyPoint : 합성곱 신경망 (CNN) 모델 구성.

<font color="green">[실습문제]</font> 2. CNN 모델을 설계해 보세요.
* 케라스를 이용해서 CNN 모델을 설계합니다.

In [43]:
# 실습해주세요.
keras.backend.clear_session()
model = Sequential()
    # Conv2D, MaxPooling2D 조합으로 층을 쌓습니다. 첫번째 입력층의 input_shape은 imageDataGenerator target size로 지정합니다.
model.add(keras.layers.Input(shape=(200,400,1)))

model.add(keras.layers.Conv2D(filters=32,padding='same',kernel_size=(3,3),strides=(1,1),activation='relu'))
model.add(keras.layers.MaxPool2D(pool_size=(2,2)))
model.add(Dropout(0.25))

# model.add(keras.layers.Conv2D(filters=32,padding='same',kernel_size=(3,3),strides=(1,1),activation='relu'))
# model.add(keras.layers.MaxPool2D(pool_size=(2,2)))
# model.add(Dropout(0.25))

model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(128,activation='relu'))
model.add(Dropout(0.25))

model.add(keras.layers.Dense(4,activation='softmax'))



In [10]:
# 설계된 모델을 확인해보세요.
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 480, 854, 32)      320       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 240, 427, 32)     0         
 )                                                               
                                                                 
 dropout (Dropout)           (None, 240, 427, 32)      0         
                                                                 
 conv2d_1 (Conv2D)           (None, 240, 427, 32)      9248      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 120, 213, 32)     0         
 2D)                                                             
                                                                 
 dropout_1 (Dropout)         (None, 120, 213, 32)      0

<font color="green">[실습문제]</font> 3. 모델을 학습시켜 보세요.
* 위에서 설계한 모델에 데이터를 가지고 학습을 진행 합니다. 
* history 변수에 학습 결과를 입력 받습니다.
* callback 함수로 ModelCheckpoint와 EarlyStopping을 사용하세요.(best만 저장)
* 학습한 모델의 weight를 경로 MODEL_PATH 에 저장해주세요.
* val_loss 기준으로 모니터링 해주세요.

In [59]:
# 실습해보세요.
# 모델 컴파일 
model.compile(loss=keras.losses.categorical_crossentropy,optimizer='adam',metrics=['Accuracy'])

# # early_stopping 
# filename=''
checkpoint = ModelCheckpoint(filepath='my_model/content/drive/MyDrive/Colab Notebooks/aivle/미니프로젝트3/AIVLE3rd_individual/model/model.ckpt',
                    #monitor='val_loss',
                    verbose=1,
                    #save_best_only=True,
                    save_weights_only=True)
# early_stopping
early_stopping = EarlyStopping(monitor='val_loss',patience=3,min_delta=0,restore_best_weights=True,verbose=1)

In [60]:
# 모델 학습
history = model.fit( 
    train_generator,
    validation_data = validation_generator, 
    epochs=10,
    verbose=1,
    callbacks=[early_stopping,checkpoint]
)


Epoch 1/10


ResourceExhaustedError: ignored

<font color="green">[실습문제]</font> 4. 모델 저장하기 
* 만들어진 모델를 기반으로 모델파일로 저장해주세요.
* 파일 저장 전에 ModelCheckpoint의 가중치(weights)를 로딩해주세요.
* 저장위치는 MODEL_PATH 입니다.
* 파일명은 <font color="red">[개인] 미니프로젝트3차_A000000_OOO.h5</font>

><font color="red">[Hint]</font><br>
>모델 가중치는 load_weight 매소드로 불러옵니다.<br>
>모델 저장시에는 model.save 매소드를 사용합니다. 

In [48]:
# 실습해보세요.

model.save('my_model/content/drive/MyDrive/Colab Notebooks/aivle/미니프로젝트3/AIVLE3rd_individual/model/model.h5')



In [47]:
tf.keras.models.save_model(model,'my_model/content/drive/MyDrive/Colab Notebooks/aivle/미니프로젝트3/AIVLE3rd_individual/model/my_model.h5')

In [37]:
new_model = keras.models.load_model('my_model/content/drive/MyDrive/Colab Notebooks/aivle/미니프로젝트3/AIVLE3rd_individual/model/model.h5')
new_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 480, 854, 32)      320       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 240, 427, 32)     0         
 )                                                               
                                                                 
 dropout (Dropout)           (None, 240, 427, 32)      0         
                                                                 
 flatten (Flatten)           (None, 3279360)           0         
                                                                 
Total params: 320
Trainable params: 320
Non-trainable params: 0
_________________________________________________________________


---

# 3. 모델 평가하기

<font color="green">[실습문제]</font> 5. 훈련 과정에서 epoch에 따른 정확도와 손실을 시각화화여 확인합니다.

In [ ]:
# 실습해보세요.
import matplotlib.pyplot as plt
%matplotlib inline








<font color="green">[실습문제]</font> 6. validation 데이터를 기준으로 학습한 모델을 적용해서 f1 Score를 계산하세요.

* <font color="red">hint.</font> sklearn.metrics 패키지에서 f1_score를 사용하세요.

In [ ]:
# 실습해보세요.
from sklearn.metrics import f1_score






---

## [추가학습] 모델 비교하기 
_시간이 남으면 해보세요._

<font color="green">[실습문제]</font> 7. 모델 구조를 변경해 보거나 다른 모델들을 만들어 보고 성능을 비교해 최고의 모델을 만들어 보세요.
- 여러분들이 배운 모델들을 다양하게 만들어 보고 성능을 비교해 보세요.
- ImageDataGenerator를 변경하면 성능 개선도 가능합니다. 

In [ ]:
# 실습해보세요.





